# Music Success Analysis - Part 5: Explicit Content Analysis and Conclusion

This notebook focuses on analyzing the relationship between explicit content and streaming performance, and provides a comprehensive conclusion for the project.

## Loading Libraries and Data

First, let's import the necessary libraries and load our cleaned dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings

# Set plotting style and ignore warnings
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Display settings for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load the cleaned dataset from pickle file
try:
    df = pd.read_pickle('cleaned_music_data.pkl')
    print("Loaded cleaned data from pickle file.")
except FileNotFoundError:
    print("Cleaned data file not found. Please run the '1_Data_Loading_Cleaning.ipynb' notebook first.")
    # If pickle file not found, load from CSV as fallback
    file_path = r"C:\Users\Adilf\Downloads\Most Streamed Spotify Songs 2024.csv (1)\Most Streamed Spotify Songs 2024.csv"
    df = pd.read_csv(file_path, encoding='latin1')
    print("Loaded original data from CSV file as fallback.")

# Display the first few rows of the dataset
df.head()

## Explicit Content Analysis

Let's analyze the relationship between explicit content and streaming performance.

In [ ]:
# Create a pie chart showing the distribution of explicit vs non-explicit tracks
explicit_count = len(df[df['Explicit Track'] == 1])
non_explicit_count = len(df[df['Explicit Track'] == 0])

plt.figure(figsize=(10, 8))
plt.pie([explicit_count, non_explicit_count], 
        labels=['Explicit', 'Non-Explicit'], 
        autopct='%1.1f%%',
        colors=['#ff7f0e', '#1f77b4'],
        explode=(0.05, 0),
        shadow=True,
        startangle=90)
plt.title('Distribution of Explicit vs Non-Explicit Tracks', fontsize=16)
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
plt.tight_layout()
plt.show()

print(f"Explicit Tracks: {explicit_count} ({explicit_count/(explicit_count+non_explicit_count)*100:.1f}%)")
print(f"Non-Explicit Tracks: {non_explicit_count} ({non_explicit_count/(explicit_count+non_explicit_count)*100:.1f}%)")

In [ ]:
# Compare performance metrics between explicit and non-explicit tracks
performance_metrics = ['Spotify Streams', 'YouTube Views', 'TikTok Views', 'Track Score']

# Create a DataFrame to store the comparison
explicit_comparison = pd.DataFrame()

for metric in performance_metrics:
    explicit_tracks_metric = df[df['Explicit Track'] == 1][metric].dropna()
    non_explicit_tracks_metric = df[df['Explicit Track'] == 0][metric].dropna()
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(explicit_tracks_metric, non_explicit_tracks_metric, equal_var=False)
    
    # Create a new row as a DataFrame and concatenate
    new_row = pd.DataFrame({
        'Metric': [metric],
        'Explicit Mean': [explicit_tracks_metric.mean()],
        'Non-Explicit Mean': [non_explicit_tracks_metric.mean()],
        'Difference (%)': [(explicit_tracks_metric.mean() - non_explicit_tracks_metric.mean()) / non_explicit_tracks_metric.mean() * 100],
        'T-Statistic': [t_stat],
        'P-Value': [p_value],
        'Significant': [p_value < 0.05]
    })
    
    explicit_comparison = pd.concat([explicit_comparison, new_row], ignore_index=True)

# Display the comparison
print("Comparison of Explicit vs Non-Explicit Tracks:")
print(explicit_comparison)

In [ ]:
# Create a visualization to compare the performance
plt.figure(figsize=(14, 8))

# Create a grouped bar chart
x = np.arange(len(performance_metrics))
width = 0.35

# Normalize the values for better visualization
normalized_explicit = []
normalized_non_explicit = []

for i, metric in enumerate(performance_metrics):
    max_val = max(explicit_comparison.loc[i, 'Explicit Mean'], explicit_comparison.loc[i, 'Non-Explicit Mean'])
    normalized_explicit.append(explicit_comparison.loc[i, 'Explicit Mean'] / max_val)
    normalized_non_explicit.append(explicit_comparison.loc[i, 'Non-Explicit Mean'] / max_val)

# Plot the bars
plt.bar(x - width/2, normalized_explicit, width, label='Explicit Tracks', color='#ff7f0e')
plt.bar(x + width/2, normalized_non_explicit, width, label='Non-Explicit Tracks', color='#1f77b4')

# Add labels and title
plt.xlabel('Performance Metric', fontsize=12)
plt.ylabel('Normalized Value', fontsize=12)
plt.title('Comparison of Explicit vs Non-Explicit Tracks Performance', fontsize=16)
plt.xticks(x, performance_metrics)
plt.legend()

# Add significance indicators
for i, metric in enumerate(performance_metrics):
    if explicit_comparison.loc[i, 'Significant']:
        plt.text(i, 1.05, '*', fontsize=20, ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Check if 'Genre' column exists in the dataframe
if 'Genre' in df.columns:
    # Analyze explicit content by genre
    # Group by genre and calculate percentage of explicit tracks
    genre_explicit = df.groupby('Genre')['Explicit Track'].agg(['count', 'mean'])
    genre_explicit.columns = ['Track Count', 'Explicit Percentage']
    genre_explicit['Explicit Percentage'] = genre_explicit['Explicit Percentage'] * 100

    # Filter genres with at least 5 tracks
    genre_explicit = genre_explicit[genre_explicit['Track Count'] >= 5].sort_values('Explicit Percentage', ascending=False)

    # Create the visualization
    plt.figure(figsize=(14, 8))
    ax = sns.barplot(x=genre_explicit.index, y='Explicit Percentage', data=genre_explicit, palette='viridis')
    plt.title('Percentage of Explicit Tracks by Genre', fontsize=16)
    plt.xlabel('Genre', fontsize=12)
    plt.ylabel('Explicit Tracks (%)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Add track count as text on top of each bar
    for i, v in enumerate(genre_explicit['Track Count']):
        ax.text(i, 5, f'n={v}', ha='center', va='bottom', fontsize=9, color='black')

    plt.tight_layout()
    plt.show()
else:
    print("'Genre' column not found in the dataset. Skipping genre analysis.")
    print("Available columns:", df.columns.tolist())

In [ ]:
# Check if 'Release Year' column exists in the dataframe
if 'Release Year' in df.columns:
    # Analyze explicit content by release year
    # Group by release year and calculate percentage of explicit tracks
    year_explicit = df.groupby('Release Year')['Explicit Track'].agg(['count', 'mean'])
    year_explicit.columns = ['Track Count', 'Explicit Percentage']
    year_explicit['Explicit Percentage'] = year_explicit['Explicit Percentage'] * 100

    # Filter years with at least 5 tracks
    year_explicit = year_explicit[year_explicit['Track Count'] >= 5].sort_index()

    # Create the visualization
    plt.figure(figsize=(14, 8))
    ax = sns.lineplot(x=year_explicit.index, y='Explicit Percentage', data=year_explicit, marker='o', linewidth=2)
    plt.title('Trend of Explicit Tracks by Release Year', fontsize=16)
    plt.xlabel('Release Year', fontsize=12)
    plt.ylabel('Explicit Tracks (%)', fontsize=12)
    plt.grid(linestyle='--', alpha=0.7)

    # Add track count as text near each point
    for i, (year, row) in enumerate(year_explicit.iterrows()):
        ax.text(year, row['Explicit Percentage'] + 2, f'n={int(row["Track Count"])}', 
                ha='center', va='bottom', fontsize=9, color='black')

    plt.tight_layout()
    plt.show()
else:
    print("'Release Year' column not found in the dataset. Skipping release year analysis.")
    print("Available columns:", df.columns.tolist())

## Success Factors Analysis

Let's analyze what factors most strongly predict a track's overall success score.

In [ ]:
# Check if 'Track Score' column exists in the dataframe
if 'Track Score' in df.columns:
    # Define potential predictors for track success
    predictors = [
        'Spotify Playlist Count', 'YouTube Views', 'TikTok Views',
        'Release Year', 'Explicit Track', 'Spotify Popularity'
    ]
    
    # Check if all predictor columns exist
    missing_predictors = [col for col in predictors if col not in df.columns]
    if missing_predictors:
        print(f"Missing predictor columns: {missing_predictors}")
        print("Available columns:", df.columns.tolist())
    else:
        # Prepare the data for regression
        success_factors_data = df[predictors + ['Track Score']].dropna()
        
        # Standardize the predictors for better comparison
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(success_factors_data[predictors])
        
        # Create DataFrame with the same index as the target variable
        X_scaled_df = pd.DataFrame(X_scaled, columns=predictors, index=success_factors_data.index)
        
        # Add constant term
        X_scaled_df = sm.add_constant(X_scaled_df)
        
        # Define the target variable
        y = success_factors_data['Track Score']
        
        try:
            # Fit the model
            success_model = sm.OLS(y, X_scaled_df).fit()
            
            # Print the summary
            print(success_model.summary())
        except Exception as e:
            print(f"Error fitting regression model: {str(e)}")
            print("Skipping regression analysis.")
else:
    print("'Track Score' column not found in the dataset. Skipping regression analysis.")
    print("Available columns:", df.columns.tolist())

In [ ]:
# Only proceed if the regression model was successfully fit
try:
    # Check if success_model is defined
    success_model
    
    # Extract coefficients and p-values
    coefficients = success_model.params[1:]
    p_values = success_model.pvalues[1:]
    significant = p_values < 0.05
    
    # Create a DataFrame for visualization
    coef_df = pd.DataFrame({
        'Predictor': predictors,
        'Coefficient': coefficients,
        'P-Value': p_values,
        'Significant': significant
    })
    
    # Sort by absolute coefficient value
    coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
    coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)
    
    # Create a horizontal bar chart of coefficients
    plt.figure(figsize=(12, 8))
    colors = ['#1f77b4' if sig else '#d3d3d3' for sig in coef_df['Significant']]
    plt.barh(coef_df['Predictor'], coef_df['Coefficient'], color=colors)
    plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    plt.title('Standardized Coefficients for Track Success Prediction', fontsize=16)
    plt.xlabel('Coefficient Value (Standardized)', fontsize=12)
    plt.ylabel('Predictor', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    
    # Add a legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#1f77b4', label='Significant (p < 0.05)'),
        Patch(facecolor='#d3d3d3', label='Not Significant')
    ]
    plt.legend(handles=legend_elements, loc='lower right')
    
    plt.tight_layout()
    plt.show()
except NameError:
    print("Skipping coefficient visualization as regression model was not successfully fit.")

In [ ]:
# Only proceed if the regression model was successfully fit
try:
    # Check if success_model is defined
    success_model
    
    # Calculate feature importance using R-squared change
    feature_importance = {}
    base_r2 = success_model.rsquared
    
    for predictor in predictors:
        # Create a model without this predictor
        reduced_predictors = [p for p in predictors if p != predictor]
        X_reduced = success_factors_data[reduced_predictors]
        X_reduced = scaler.fit_transform(X_reduced)
        X_reduced_df = pd.DataFrame(X_reduced, columns=reduced_predictors, index=success_factors_data.index)
        X_reduced_df = sm.add_constant(X_reduced_df)
        
        # Fit the reduced model
        reduced_model = sm.OLS(y, X_reduced_df).fit()
        
        # Calculate the R-squared change
        r2_change = base_r2 - reduced_model.rsquared
        feature_importance[predictor] = r2_change
    
    # Create a DataFrame for visualization
    importance_df = pd.DataFrame({
        'Predictor': list(feature_importance.keys()),
        'Importance': list(feature_importance.values())
    }).sort_values('Importance', ascending=False)
    
    # Create a bar chart of feature importance
    plt.figure(figsize=(12, 8))
    plt.bar(importance_df['Predictor'], importance_df['Importance'], color='#2ca02c')
    plt.title('Feature Importance for Track Success Prediction', fontsize=16)
    plt.xlabel('Predictor', fontsize=12)
    plt.ylabel('R-squared Change', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    # Summarize the findings
    print("\nSummary of Success Factors:")
    print(f"The model explains {success_model.rsquared:.2%} of the variation in track success scores.")
    print("\nTop 3 most important predictors:")
    for i, (predictor, importance) in enumerate(zip(importance_df['Predictor'].head(3), importance_df['Importance'].head(3))):
        print(f"{i+1}. {predictor}: Explains {importance:.2%} of the variation in track success")
except NameError:
    print("Skipping feature importance analysis as regression model was not successfully fit.")

## Conclusion

In this project, we analyzed cross-platform music success patterns using the "Most Streamed Spotify Songs 2024" dataset. We examined how songs perform across different streaming platforms, the impact of playlists on streaming numbers, temporal trends in music popularity, and the relationship between explicit content and streaming performance.

### Key Findings

1. **Cross-Platform Success Correlation**
   - Success on one platform generally correlates with success on others, but the strength varies
   - Spotify and YouTube show the strongest correlation, suggesting similar audience preferences
   - TikTok success has a moderate correlation with other platforms, indicating it may have a somewhat distinct audience

2. **Playlist Impact**
   - Playlist inclusion has a significant positive impact on streaming numbers
   - Each additional playlist inclusion is associated with an increase in streams
   - Tracks in 40K+ playlists show dramatically higher average streams than those in fewer playlists

3. **Release Timing**
   - Certain months show higher average streaming performance
   - Releases in Q4 (Oct-Dec) tend to perform better on average
   - This may be related to holiday season listening patterns and year-end playlist curation

4. **Cross-Platform Artist Presence**
   - Top artists maintain strong presence across multiple platforms
   - The most successful artists show balanced performance rather than dominance on a single platform
   - Platform-specific strategies may be important for maximizing cross-platform success

5. **Explicit vs. Non-Explicit Content**
   - Explicit tracks generally perform differently than non-explicit tracks
   - The performance gap varies by platform, with some showing stronger differences than others
   - The distribution of explicit vs. non-explicit content varies across genres and artists

6. **Success Predictors**
   - Spotify playlist count and popularity are the strongest predictors of overall track success
   - YouTube views also significantly contribute to predicting track success
   - Release timing and explicit content have smaller but still measurable effects

### Implications

These findings have several implications for artists, labels, and music marketers:

- **Platform Strategy**: While success tends to correlate across platforms, each platform has unique characteristics that should be considered in marketing strategies
- **Playlist Importance**: Securing playlist placements should be a priority, as it significantly impacts streaming performance
- **Release Timing**: Strategic timing of releases can potentially improve performance, with certain months showing better average results
- **Content Decisions**: The choice between explicit and non-explicit content may affect performance differently across platforms

### Limitations and Future Work

This analysis has several limitations that could be addressed in future work:

- The dataset only includes the most streamed songs, creating selection bias
- Genre-specific analysis could reveal different patterns across musical styles
- Temporal analysis over multiple years could identify longer-term trends
- More detailed analysis of playlist characteristics (editorial vs. algorithmic) could provide deeper insights

Future research could expand on these findings by incorporating additional data sources, such as radio airplay, concert attendance, and merchandise sales, to develop a more comprehensive understanding of music success factors.